**Environment:** the first code cells print the exact library versions this notebook ran with (`transformers_version=` / `torch_version=` key=value lines; `fla_version=` records `not-used` - no flash-linear-attention dependency here) and install the FEASIBILITY-locked megaDNA prerequisites: the `lingxusb/megaDNA` clone pinned to commit `cb2f5ab4cc88dc0effe05c5f23358862c837014a` plus `MEGABYTE_pytorch==0.2.1` - never a floating clone or a floating MEGABYTE version. The install targets the running kernel's `VIRTUAL_ENV` and is reversible (exact versions).

In [ ]:
# Provenance stamp (D-21): the exact versions this notebook ran with, printed
# as key=value lines. No flash-linear-attention dependency here, so the fla
# line records not-used and the FEASIBILITY-locked megaDNA pins are stamped
# alongside (the pinned install cell below).
import torch
import transformers

print(f"transformers_version={transformers.__version__}")
print(f"torch_version={torch.__version__}")
print("fla_version=not-used")
print("megadna_commit=cb2f5ab4cc88dc0effe05c5f23358862c837014a")
print("megabyte_version=0.2.1")

In [ ]:
# Install the pinned megaDNA prerequisites (FEASIBILITY lock): the clone is
# pinned to commit cb2f5ab4cc88dc0effe05c5f23358862c837014a and MEGABYTE_pytorch to 0.2.1 (the repo's own
# requirements pin) - never a floating clone or a floating MEGABYTE version.
# "uv pip install" targets the VIRTUAL_ENV of the running kernel: the project
# venv for this notebook (the pinned install is reversible - exact versions).
!git clone https://github.com/lingxusb/megaDNA.git
!git -C megaDNA checkout cb2f5ab4cc88dc0effe05c5f23358862c837014a
!uv pip install MEGABYTE_pytorch==0.2.1 ./megaDNA

In [2]:
from dnallm import load_config
from dnallm.models import load_model_and_tokenizer
from dnallm.inference import DNAInference

In [3]:
# Load configurations
configs = load_config("./inference_megaDNA_config.yaml")

In [ ]:
model_name = "lingxusb/megaDNA_updated"
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs['task'], source="huggingface")

2025-10-10 02:38:15,906 - modelscope - INFO - Not logged-in, you can login for uploadingor accessing controlled entities.


02:38:16 - dnallm.models.model - INFO - Model files are stored in /home/liuguanqing/.cache/modelscope/hub/models/lgq12697/megaDNA_updated


In [5]:
# Create inference engine
inference_engine = DNAInference(
    model=model,
    tokenizer=tokenizer,
    config=configs
)

02:38:17 - dnallm.models.model - INFO - Using device: cuda


In [6]:
output = inference_engine.generate(["ACGT"], n_tokens=1024, temperature=0.95, top_p=0.1)

  0%|          | 0/1020 [00:00<?, ?it/s]

/home/liuguanqing/miniforge3/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


In [7]:
for seq in output:
    print(f"Input Sequence: {seq['Prompt']}")
    print(f"Generated Sequence: {seq['Output']}")
    print()

Input Sequence: ACGT
Generated Sequence: ACGTGTGGCTCGAATATGGGCAGATTGAGCAGCTTGAGCGGCTCGTAAATGAAAGCGGAATTGACATTGCAAATACCGACGACCACTATTACTTGAAAATTGATGAGGTGTAAATGATGGACGCTACAATGAACTATAACACGTTGCCCTATCTTTCCAAGGCGCAGAACGGAGTGACTGAGAGGCTCCAAAAGGGAGACATCACGCAGAAAGATATCTTGCAGGACATTTTGTCAGACGGTTTCTTTTTTTTGAAGATTAAAGGGCGGAGCTTTGCGGATGCGTCTGAAAAACTGGCTCATCATGGGCGAAAAAAGACAGTCGCTAATATTGACATCTCAAAGTTGTTAAATGCGGCGGAAGATAAGGGCTTTATGAACCGTTACCTAAAGCTCGTGGATTCGCTACAGAGAATAATCAAGGGAGGGGATGTTGGAATCTTGGACGGCTATCAAAACGGCACTGAATTCGACAGCGTATTGGCTAGATGCATTGACAAGTTTAATATACAAGACTTCATGAGACCTGCAAACGAGGGAAGAGGCGTGGAAAGTATATCTTTGACGGCTTTGCTTACCCGTCAAAAGCAAAAAGCCAAGTATTCCAAGATGCTTCTGCGAATGCCGGCTGCGGAATTGAAAAATGAATTGGCAAAGCTGTGCCCCGACTGCAAAGATAGAAAAGAGAAATACAAATTACCCAGATGGTTAGCTTCAAATAATAAAGATGCATGGCTTTATTCTTATATGCCTGATGTGGGCAAACGTTGGAATGTCAATAATGCGACGCTTGTAGAATGAGGAATTAACAATATCAGAGGGAGGATTCCCGTAGCCGCCTTTCTTATTTCTTTCGATACCGAATGAAATCGATGATCTCACCGGGCTTGAGCTTGACGATGCAGGACTGGATTTTGGAGATTCCCTGTCGGCGCATGATTCCGACATTGAGGCAGCGCA

In [8]:
scores = inference_engine.scoring(["ATCCGCATG", "ATGCGCATG"])
for res in scores:
    print(f"Input Sequence: {res['Input']}")
    print(f"Score: {res['Score']}")
    print()

Input Sequence: ATCCGCATG
Score: 1.4011831283569336

Input Sequence: ATGCGCATG
Score: 1.4185738563537598

